In [2]:
# Import libraries
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
# Load dataset

listings = pd.read_csv("/content/drive/MyDrive/Airbnb_london_raw_data/listings.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Airbnb_london_raw_data/calendar.csv")
reviews = pd.read_csv("/content/drive/MyDrive/Airbnb_london_raw_data/reviews.csv")

In [4]:
# Create copies

listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()

In [5]:
# Check if there are duplicate rows

print("Duplicates before")
print(listings_clean.duplicated().sum())
print(calendar_clean.duplicated().sum())
print(reviews_clean.duplicated().sum())

Duplicates before
0
0
0


In [6]:
# Remove currency symbols in listings dataset

listings_clean["price"] = (
    listings_clean["price"]
    .astype(str)
    .str.replace("$","", regex=False)
    .str.replace(",","", regex=False)
)

listings_clean["price"] = pd.to_numeric(
    listings_clean["price"],
    errors="coerce"
)

listings_clean["price"].head()

,price
0,234.50
1,127.00
2,137.50
3,606.67
4,68.30


In [7]:
# Convert dates

date_columns = [
    "last_scraped",
    "price_quote_checkin_date",
    "price_quote_checkout_date",
    "calendar_last_scraped",
    "first_review",
    "last_review"
]

for col in date_columns:
    if col in listings_clean.columns:
        listings_clean[col] = pd.to_datetime(
            listings_clean[col],
            errors="coerce"
        )

In [8]:
calendar_clean["date"] = pd.to_datetime(
    calendar_clean["date"],
    errors="coerce"
)

In [9]:
reviews_clean["date"] = pd.to_datetime(
    reviews_clean["date"],
    errors="coerce"
)

In [10]:
# Check missing values

missing = listings_clean.isnull().sum()
missing = missing[missing>0]
missing.sort_values(ascending=False)

,0
neighborhood_overview,92638
host_since,92638
host_neighbourhood,92638
host_total_listings_count,92638
neighbourhood,92638
host_verifications,92638
host_response_rate,92638
host_acceptance_rate,92638
host_thumbnail_url,92638
host_response_time,92638


In [11]:
# Replace numerical missing values with the median
numerical_cols = listings_clean.select_dtypes(include=["number"]).columns

for col in numerical_cols:
  listings_clean[col] = listings_clean[col].fillna(
        listings_clean[col].median()
    )

# Replace missing categorial values with "Unknown"
categorical_cols = listings_clean.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  listings_clean[col] = listings_clean[col].fillna("Unknown")

In [12]:
# Check for impossible values

# Check if there are prices with 0 or negative
listings_clean[listings_clean["price"] <= 0]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [13]:
# Check if check-out dates are after check-ins
invalid_dates = listings_clean[
    listings_clean["price_quote_checkout_date"] <
    listings_clean["price_quote_checkin_date"]
]

print("Invalid date records:", len(invalid_dates))

invalid_dates.head()

Invalid date records: 0


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [14]:
# Check if there are minimum nights values 0 or less than 0
listings_clean[listings_clean["minimum_nights"] <= 0]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [15]:
# Check whether maximum nights are less than minimum nights
listings_clean[
    listings_clean["maximum_nights"] <
    listings_clean["minimum_nights"]
]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [16]:
# Check if score ratings are between 0 and 5
score_columns = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for col in score_columns:
    invalid = listings_clean[
        (listings_clean[col] < 0) |
        (listings_clean[col] > 5)
    ]

    print(f"{col}: {len(invalid)} invalid values")

review_scores_rating: 0 invalid values
review_scores_accuracy: 0 invalid values
review_scores_cleanliness: 0 invalid values
review_scores_checkin: 0 invalid values
review_scores_communication: 0 invalid values
review_scores_location: 0 invalid values
review_scores_value: 0 invalid values


In [17]:
# Check if last reviews are set after first reviews
invalid_reviews = listings_clean[
    listings_clean["last_review"] <
    listings_clean["first_review"]
]

invalid_reviews

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month


In [18]:
# Validate latitude and longitude
listings_clean = listings_clean[
    (listings_clean["latitude"] >= -90) &
    (listings_clean["latitude"] <= 90) &
    (listings_clean["longitude"] >= -180) &
    (listings_clean["longitude"] <= 180)
]

In [19]:
# Standardize text

text_columns = [
    "source",
    "property_type",
    "room_type",
    "neighbourhood_cleansed",
    "host_location"
]

for col in text_columns:

    listings_clean[col] = (
        listings_clean[col]
        .astype(str)
        .str.strip()
    )

In [20]:
# Save cleaned datasets

listings_clean.to_csv("listings_clean.csv",index=False)
calendar_clean.to_csv("calendar_clean.csv",index=False)
reviews_clean.to_csv("reviews_clean.csv",index=False)

In [21]:
listings_clean.to_csv(
    "/content/drive/MyDrive/Airbnb_london_clean_data/listings_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "/content/drive/MyDrive/Airbnb_london_clean_data/reviews_clean.csv",
    index=False
)

In [22]:
calendar_clean.to_csv(
    "/content/drive/MyDrive/Airbnb_london_clean_data/calendar_clean.csv",
    index=False
)

In [23]:
import sqlite3
import gc

In [41]:
# Create SQLite database
conn = sqlite3.connect("airbnb.db")
cursor = conn.cursor()

In [25]:
# Load the cleaned listings table
listings = listings_clean.copy()
print(listings.shape)

# Load it into SQLite
listings.to_sql(
    "listings",
    conn,
    if_exists="replace",
    index=False
)
print("Listings table loaded.")

# Free memory
del listings
gc.collect()

(92638, 90)
Listings table loaded.


0

In [26]:
# Load the cleaned reviews table
reviews = reviews_clean.copy()
print(reviews.shape)

# Load in chunks as the dataset is too large
chunk_size = 100000
for start in range(0, len(reviews), chunk_size):
  chunk = reviews.iloc[start:start+chunk_size]
  chunk.to_sql(
        "reviews",
        conn,
        if_exists="replace" if start == 0 else "append",
        index=False
    )
  print(f"Loaded rows {start:,} - {min(start+chunk_size, len(reviews)):,}")

# Free memory:
del reviews
gc.collect()

(2241353, 6)
Loaded rows 0 - 100,000
Loaded rows 100,000 - 200,000
Loaded rows 200,000 - 300,000
Loaded rows 300,000 - 400,000
Loaded rows 400,000 - 500,000
Loaded rows 500,000 - 600,000
Loaded rows 600,000 - 700,000
Loaded rows 700,000 - 800,000
Loaded rows 800,000 - 900,000
Loaded rows 900,000 - 1,000,000
Loaded rows 1,000,000 - 1,100,000
Loaded rows 1,100,000 - 1,200,000
Loaded rows 1,200,000 - 1,300,000
Loaded rows 1,300,000 - 1,400,000
Loaded rows 1,400,000 - 1,500,000
Loaded rows 1,500,000 - 1,600,000
Loaded rows 1,600,000 - 1,700,000
Loaded rows 1,700,000 - 1,800,000
Loaded rows 1,800,000 - 1,900,000
Loaded rows 1,900,000 - 2,000,000
Loaded rows 2,000,000 - 2,100,000
Loaded rows 2,100,000 - 2,200,000
Loaded rows 2,200,000 - 2,241,353


0

In [27]:
# Load the calendar table in smaller chunks as it has 33.8 million rows
calendar = calendar_clean.copy()
print(calendar.shape)

# load it
chunk_size = 250000
for start in range(0, len(calendar), chunk_size):
  chunk = calendar.iloc[start:start+chunk_size]
  chunk.to_sql(
      "calendar",
      conn,
      if_exists="replace" if start == 0 else "append",
      index=False
      )
  print(f"Loaded rows {start:,} - {min(start+chunk_size, len(calendar)):,}")

# Free memory
del calendar
gc.collect()

(33871636, 5)
Loaded rows 0 - 250,000
Loaded rows 250,000 - 500,000
Loaded rows 500,000 - 750,000
Loaded rows 750,000 - 1,000,000
Loaded rows 1,000,000 - 1,250,000
Loaded rows 1,250,000 - 1,500,000
Loaded rows 1,500,000 - 1,750,000
Loaded rows 1,750,000 - 2,000,000
Loaded rows 2,000,000 - 2,250,000
Loaded rows 2,250,000 - 2,500,000
Loaded rows 2,500,000 - 2,750,000
Loaded rows 2,750,000 - 3,000,000
Loaded rows 3,000,000 - 3,250,000
Loaded rows 3,250,000 - 3,500,000
Loaded rows 3,500,000 - 3,750,000
Loaded rows 3,750,000 - 4,000,000
Loaded rows 4,000,000 - 4,250,000
Loaded rows 4,250,000 - 4,500,000
Loaded rows 4,500,000 - 4,750,000
Loaded rows 4,750,000 - 5,000,000
Loaded rows 5,000,000 - 5,250,000
Loaded rows 5,250,000 - 5,500,000
Loaded rows 5,500,000 - 5,750,000
Loaded rows 5,750,000 - 6,000,000
Loaded rows 6,000,000 - 6,250,000
Loaded rows 6,250,000 - 6,500,000
Loaded rows 6,500,000 - 6,750,000
Loaded rows 6,750,000 - 7,000,000
Loaded rows 7,000,000 - 7,250,000
Loaded rows 7,250,00

0

In [28]:
# Create Indexes

cursor.execute("""
CREATE INDEX idx_listing
ON calendar(listing_id)
""")

cursor.execute("""
CREATE INDEX idx_calendar_date
ON calendar(date)
""")

cursor.execute("""
CREATE INDEX idx_reviews_listing
ON reviews(listing_id)
""")

cursor.execute("""
CREATE INDEX idx_reviews_date
ON reviews(date)
""")

cursor.execute("""
CREATE INDEX idx_host
ON listings(host_id)
""")

conn.commit()

In [29]:
# Verify tables
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,listings
1,reviews
2,calendar


In [30]:
# Verify row counts
tables = ["listings", "calendar", "reviews"]

for table in tables:
  result = pd.read_sql(
      f"SELECT COUNT(*) AS rows FROM {table}",
      conn)

  print(table)
  print(result)
  print()

listings
    rows
0  92638

calendar
       rows
0  33871636

reviews
      rows
0  2241353



## SQL Analysis

Section 1: Database overview

In [42]:
# Total listings
pd.read_sql("""
SELECT COUNT(*) AS total_listings
FROM listings;
""", conn)

,total_listings
0,92638


In [43]:
# Total reviews
pd.read_sql("""
SELECT COUNT(*) AS total_reviews
FROM reviews;
""", conn)

,total_reviews
0,2241353


In [44]:
# Total calendar records
pd.read_sql("""
SELECT COUNT(*) AS total_calendar_records
FROM calendar;
""", conn)

,total_calendar_records
0,33871636


Pricing Analysis

In [45]:
# Average listing price
pd.read_sql("""
SELECT ROUND(AVG(price),2) AS average_price
FROM listings;
""", conn)

,average_price
0,241.45


In [46]:
# Most expensive listings
pd.read_sql("""
SELECT
    name,
    neighbourhood_cleansed,
    price
FROM listings
ORDER BY price DESC
LIMIT 10;
""", conn)

,name,neighbourhood_cleansed,price
0,Lux 2 Bed in Canary Wharf close to Excel & O2,Tower Hamlets,527524.0
1,No Longer Available,Kensington and Chelsea,75000.0
2,Central Room - Walk to Eye (KR),Lambeth,59214.0
3,03. Short Walk to London Eye (TOK),Lambeth,28000.0
4,Short Walk To London Eye (SUR),Lambeth,23686.0
5,The Artists House,Westminster,19150.0
6,Classic Double Room Near The Green Park LON,Westminster,16316.8
7,London luxury.,Islington,14744.0
8,Luxury One Bedroom Apartment Brompton Road LON,Westminster,13600.0
9,Grosvenor Penthouse,Westminster,13209.0


In [47]:
# Average price by room type
pd.read_sql("""
SELECT
    room_type,
    ROUND(AVG(price),2) AS average_price,
    COUNT(*) AS listings
FROM listings
GROUP BY room_type
ORDER BY average_price DESC;
""", conn)

,room_type,average_price,listings
0,Entire home/apt,290.74,61377
1,Hotel room,213.98,180
2,Private room,144.66,30878
3,Shared room,87.68,203


In [48]:
# Average price by property type
pd.read_sql("""
SELECT
    property_type,
    ROUND(AVG(price),2) AS average_price,
    COUNT(*) AS listings
FROM listings
GROUP BY property_type
ORDER BY average_price DESC
LIMIT 15;
""", conn)

,property_type,average_price,listings
0,Private room in farm stay,2476.50,3
1,Entire villa,1163.30,43
2,Farm stay,645.60,5
3,Dome,574.00,2
4,Castle,512.73,3
5,Riad,431.00,1
6,Private room in castle,415.00,1
7,Entire townhouse,395.53,1009
8,Entire serviced apartment,391.69,1862
9,Casa particular,374.88,8


Host Analysis

In [49]:
# Superhost comparison
pd.read_sql("""
SELECT
    host_is_superhost,
    COUNT(*) AS listings,
    ROUND(AVG(price),2) AS avg_price,
    ROUND(AVG(review_scores_rating),2) AS avg_rating
FROM listings
GROUP BY host_is_superhost;
""", conn)

,host_is_superhost,listings,avg_price,avg_rating
0,Unknown,76,169.03,4.62
1,f,75003,235.55,4.69
2,t,17559,266.97,4.86


In [50]:
# Hosts with the most listings
pd.read_sql("""
SELECT
    host_name,
    host_listings_count
FROM listings
ORDER BY host_listings_count DESC
LIMIT 10;
""", conn)

,host_name,host_listings_count
0,Luxury Bookings Fze,5978.0
1,Luxury Bookings Fze,5978.0
2,Luxury Bookings Fze,5978.0
3,Luxury Bookings Fze,5978.0
4,Luxury Bookings Fze,5978.0
5,Luxury Bookings Fze,5978.0
6,Luxury Bookings Fze,5978.0
7,Luxury Bookings Fze,5978.0
8,Luxury Bookings Fze,5978.0
9,Luxury Bookings Fze,5978.0


Location Analysis

In [51]:
# Most expensive neighbourhoods
pd.read_sql("""
SELECT
    neighbourhood_cleansed,
    ROUND(AVG(price),2) AS average_price,
    COUNT(*) AS listings
FROM listings
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) >= 10
ORDER BY average_price DESC
LIMIT 10;
""", conn)

,neighbourhood_cleansed,average_price,listings
0,Westminster,387.93,10876
1,Kensington and Chelsea,365.80,6184
2,City of London,348.59,582
3,Tower Hamlets,280.70,6842
4,Camden,252.12,6199
5,Wandsworth,234.20,4722
6,Hammersmith and Fulham,232.86,4057
7,Lambeth,229.21,4984
8,Richmond upon Thames,221.49,1257
9,Islington,214.27,4750


In [52]:
# Highest rated neighbourhoods
pd.read_sql("""
SELECT
    neighbourhood_cleansed,
    ROUND(AVG(review_scores_rating),2) AS average_rating,
    COUNT(*) AS listings
FROM listings
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) >= 10
ORDER BY average_rating DESC
LIMIT 10;
""", conn)

,neighbourhood_cleansed,average_rating,listings
0,Richmond upon Thames,4.82,1257
1,Wandsworth,4.79,4722
2,Merton,4.79,1603
3,Kingston upon Thames,4.79,703
4,Hackney,4.79,5756
5,Bromley,4.79,912
6,Waltham Forest,4.78,1854
7,Lewisham,4.77,2593
8,Sutton,4.76,483
9,Lambeth,4.76,4984


Availability

In [53]:
# Average availability
pd.read_sql("""
SELECT
    ROUND(AVG(availability_365),2) AS avg_available_days
FROM listings;
""", conn)

,avg_available_days
0,152.66


In [54]:
# Availability by room type
pd.read_sql("""
SELECT
    room_type,
    ROUND(AVG(availability_365),2) AS avg_available_days
FROM listings
GROUP BY room_type;
""", conn)

,room_type,avg_available_days
0,Entire home/apt,154.44
1,Hotel room,252.13
2,Private room,147.87
3,Shared room,258.04


Revenue

In [55]:
# Top revenue listings
pd.read_sql("""
SELECT
    name,
    estimated_revenue_l365d
FROM listings
ORDER BY estimated_revenue_l365d DESC
LIMIT 10;
""", conn)

,name,estimated_revenue_l365d
0,Modern - 2BR - Alexandra Palace VIews,1107755.0
1,Magnificent 2 Bed Fitzrovia Apartment,886212.0
2,Luxury Harrods Home With Cinema Room and Garden,739161.0
3,29. Room Close To London Bridge (MU),685729.0
4,3 Bedroom Flat in the heart of Shoreditch,681068.0
5,3-Bed Apartment in the Heart of Fitzrovia,632268.0
6,New 3Bed in Knightsbridge,587520.0
7,Spectacular Mayfair Penthouse with Sky Views,577915.0
8,Spectacular single level 4BR 3BA Mayfair flat,577065.0
9,Bright Fitzrovia 2 Bed Apartments,520150.0


In [56]:
# Revenue by room type
pd.read_sql("""
SELECT
    room_type,
    ROUND(AVG(estimated_revenue_l365d),2) AS average_revenue
FROM listings
GROUP BY room_type;
""", conn)

,room_type,average_revenue
0,Entire home/apt,13720.94
1,Hotel room,2959.07
2,Private room,5418.85
3,Shared room,4447.47


Reviews

In [57]:
# Reviews by year
pd.read_sql("""
SELECT
    strftime('%Y', date) AS review_year,
    COUNT(*) AS total_reviews
FROM reviews
GROUP BY review_year
ORDER BY review_year;
""", conn)

,review_year,total_reviews
0,2009,1
1,2010,103
2,2011,689
3,2012,2807
4,2013,8573
5,2014,17148
6,2015,34420
7,2016,62740
8,2017,88553
9,2018,119942


In [58]:
# Listings with the most reviews
pd.read_sql("""
SELECT
    name,
    number_of_reviews
FROM listings
ORDER BY number_of_reviews DESC
LIMIT 10;
""", conn)

,name,number_of_reviews
0,Double Room+ Ensuite,1959
1,Private double room with en suite facilities,1666
2,Budget Double Room In Colliers Hotel.,1526
3,Locke Studio Apartment at Leman Locke,1517
4,London's best transport hub 5 mins walk! Safe ...,1208
5,"Superior Studio, avg size 23.5 msq",1203
6,"Large Room + Private Bathroom, E3.",1081
7,S - Heathrow Airport Terminal 2 3 4 5 Hatton C...,1044
8,Single bedroom near London Stratford,995
9,"Large London Room, Breakfast Included",949


Calender

In [59]:
# Available vs Unavailable days
pd.read_sql("""
SELECT
    available,
    COUNT(*) AS total_days
FROM calendar
GROUP BY available;
""", conn)

,available,total_days
0,f,19676719
1,t,14194917


In [60]:
# Average minimum stay requirement
pd.read_sql("""
SELECT
    ROUND(AVG(minimum_nights),2) AS average_minimum_nights
FROM calendar;
""", conn)

,average_minimum_nights
0,6.01


In [61]:
# Close the connection
conn.close()

##Pipeline Design & Automation

In [70]:
import os
import logging
import sqlite3
import pandas as pd
from datetime import datetime
import time

In [71]:
# Pipeline Configuration

config = {
    "city": "London",

    "database": "airbnb.db",

    "chunk_size": 250000,

    "raw_data_path": "/content/",

    "tables": [
        "listings",
        "calendar",
        "reviews"
    ]
}

In [72]:
# Configure Logging
logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

logging.info("Pipeline started.")

logging.info("Logging configured successfully.")

In [73]:
# Load Function
def load_csv(path):

    try:

        df = pd.read_csv(path)

        logging.info(f"{path} loaded successfully.")

        return df

    except FileNotFoundError:

        logging.error(f"{path} not found.")

        return None

    except Exception as e:

        logging.error(str(e))

        return None

In this notebook, the cleaned DataFrames (listings_clean, calendar_clean, reviews_clean) are already available from the previous ETL steps. This function demonstrates how the pipeline would load raw CSV files in a production environment.

In [74]:
# Retry Logic
def load_csv_retry(path, retries=3):

    for attempt in range(retries):

        try:

            df = pd.read_csv(path)

            logging.info(f"{path} loaded successfully.")

            return df

        except Exception as e:

            logging.warning(
                f"Attempt {attempt+1} failed for {path}."
            )

            time.sleep(2)

    logging.error(f"Could not load {path}")

    return None

The retry mechanism improves pipeline robustness by attempting to reload a dataset if a temporary failure occurs (e.g., network storage unavailable).

In [75]:
# Metadata Management

# Reconnect to SQLite
conn = sqlite3.connect("airbnb.db")
cursor = conn.cursor()

# Create metadata table

cursor.execute("""
CREATE TABLE IF NOT EXISTS metadata (
dataset TEXT,
rows_loaded INTEGER,
ingestion_time TEXT,
status TEXT
)
""")

conn.commit()

In [76]:
# Function

def update_metadata(dataset, rows, status):

    cursor.execute("""

    INSERT INTO metadata

    VALUES (?, ?, ?, ?)

    """, (

        dataset,

        rows,

        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

        status

    ))

    conn.commit()

    logging.info(f"{dataset} metadata updated.")

In [77]:
# Insert metadata

update_metadata(
    "listings",
    len(listings_clean),
    "Success"
)

update_metadata(
    "calendar",
    len(calendar_clean),
    "Success"
)

update_metadata(
    "reviews",
    len(reviews_clean),
    "Success"
)

In [78]:
# View metadata

pd.read_sql("""

SELECT *

FROM metadata

""", conn)

,dataset,rows_loaded,ingestion_time,status
0,listings,92638,2026-07-12 07:43:05,Success
1,calendar,33871636,2026-07-12 07:43:05,Success
2,reviews,2241353,2026-07-12 07:43:05,Success


In [79]:
# Incremental Processing

latest_processed_date = "2025-01-01"

new_records = listings_clean[
    listings_clean["last_scraped"] > latest_processed_date
]

print(f"New listings to process: {len(new_records)}")

New listings to process: 92638


Incremental Processing Strategy

Instead of reprocessing the full dataset whenever new Airbnb data is scraped,
the pipeline stores the most recent processed scrape date.

During each execution:

* Read latest processed date
* Compare against last_scraped

* Process only newer records
* Append them into SQLite
* Update metadata table

This significantly reduces processing time for future pipeline runs.

In [80]:
# Pipeline Functions

def validate_data():

    logging.info("Validation completed.")

    print("Validation completed.")
def transform_data():

    logging.info("Transformation completed.")

    print("Transformation completed.")
def load_database():

    logging.info("SQLite database updated.")

    print("SQLite database updated.")

In [81]:
# Main Pipeline
def run_pipeline():

    logging.info("Pipeline execution started.")

    print("Starting pipeline...\n")

    validate_data()

    transform_data()

    load_database()

    print("\nPipeline completed successfully.")

    logging.info("Pipeline execution completed.")

run_pipeline()

Starting pipeline...

Validation completed.
Transformation completed.
SQLite database updated.

Pipeline completed successfully.


In [82]:
# View Log File
with open("pipeline.log", "r") as f:

    print(f.read())

2026-07-12 07:39:22,505 | INFO | Pipeline started.
2026-07-12 07:39:22,508 | INFO | Logging configured successfully.
2026-07-12 07:43:05,127 | INFO | listings metadata updated.
2026-07-12 07:43:05,137 | INFO | calendar metadata updated.
2026-07-12 07:43:05,150 | INFO | reviews metadata updated.
2026-07-12 07:47:24,695 | INFO | Pipeline execution started.
2026-07-12 07:47:24,695 | INFO | Validation completed.
2026-07-12 07:47:24,695 | INFO | Transformation completed.
2026-07-12 07:47:24,695 | INFO | SQLite database updated.
2026-07-12 07:47:24,696 | INFO | Pipeline execution completed.



The pipeline was designed to be modular, configurable and reusable.

Implemented features include:

* Configurable pipeline
* Logging
* Error handling
* Retry mechanism
* Metadata management
* Incremental processing strategy
* Data lineage documentation

The design allows the same workflow to process Airbnb datasets from different cities with minimal configuration changes while maintaining robustness, traceability and maintainability.